Building an Aviation Operations Agent with ReAct from scratch

#Imports and OpenAI setup

In [1]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()

from openai import OpenAI

In [2]:
client = OpenAI(api_key = os.getenv('OPENAI_API_KEY'))

In [5]:
chat_completion = client.chat.completions.create(
    model = "gpt-4o",
    temperature = 0,
    messages = [{"role": "user", "content": "Hello world"}]
)
print(chat_completion.choices[0].message.content)

Hello! How can I assist you today?


Agent Class

In [6]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []

        if self.system:
            self.messages.append({
                "role": "system",
                "content": system
            })

    def __call__(self, message):
        self.messages.append({
            "role": "user",
            "content": message
        })

        result = self.execute()

        self.messages.append({
            "role": "assistant",
            "content": result
        })

        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="gpt-4o",
            temperature=0,
            messages=self.messages
        )

        return completion.choices[0].message.content

Our ReAct prompt

In [7]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.

Use Thought to describe what information you need and what you intend to do.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

get_flight_delay:
e.g. get_flight_delay: FL101
Returns the delay in minutes for a flight.

get_flight_distance:
e.g. get_flight_distance: FL101
Returns the flight distance in kilometers.

calculate:
e.g. calculate: 45 + 35
Runs a calculation and returns the number.
Use floating point syntax when necessary.

Example session:

Question: What is the delay of flight FL101?

Thought: I need to find the delay of FL101.
Action: get_flight_delay: FL101
PAUSE

You will be called again with:

Observation: Flight FL101 was delayed by 45 minutes.

You then output:

Answer: Flight FL101 was delayed by 45 minutes.
""".strip()

Synthetic flight data for manual testing

In [8]:
flights = {
    "FL101": {
        "delay": 45,
        "distance": 2150
    },
    "FL202": {
        "delay": 35,
        "distance": 1800
    },
    "FL303": {
        "delay": 20,
        "distance": 1250
    }
}

Define the tools

In [9]:
#Flight Delay tool
def get_flight_delay(flight):
    flight = flight.strip().upper()

    if flight in flights:
        return f"Flight {flight} was delayed by {flights[flight]['delay']} minutes."
    else:
        return f"No data available for flight {flight}."

#Flight distance tool
def get_flight_distance(flight):
    flight = flight.strip().upper()

    if flight in flights:
        return f"Flight {flight} covers a distance of {flights[flight]['distance']} km."
    else:
        return f"No distance data available for flight {flight}."

#Calculator tool
def calculate(what):
    return eval(what)

Register the tools.

This is the bridge between what the LLM requests and the Python functions that actually execute.

In [10]:
known_actions = {
    "get_flight_delay": get_flight_delay,
    "get_flight_distance": get_flight_distance,
    "calculate": calculate
}

Test the tools individually

In [11]:
print(get_flight_delay("FL101"))
print(get_flight_distance("FL101"))
print(calculate("45 + 35"))

Flight FL101 was delayed by 45 minutes.
Flight FL101 covers a distance of 2150 km.
80


Test the Agent directly

In [12]:
abot = Agent(prompt)

result = abot("What is the delay of flight FL101?")
print(result)

Thought: I need to find the delay of flight FL101.
Action: get_flight_delay: FL101
PAUSE


In [13]:
"""
Notice that the model doesn't execute the function itself.

It only tells our application:

"I want get_flight_delay with input FL101."
"""

'\nNotice that the model doesn\'t execute the function itself.\n\nIt only tells our application:\n\n"I want get_flight_delay with input FL101."\n'

Manually provide the observation

This is useful for understanding the mechanics before automating the loop.

In [14]:
result = get_flight_delay("FL101")

print("Observation:", result)

next_prompt = "Observation: {}".format(result)

abot(next_prompt)

Observation: Flight FL101 was delayed by 45 minutes.


'Answer: Flight FL101 was delayed by 45 minutes.'

The LLM now receives the tool result and should produce the final answer.

Test a multi-step interaction manually

Now let's make the agent solve something that requires multiple tools

In [15]:
abot = Agent(prompt)

question = """
Flight FL101 was delayed and Flight FL202 was also delayed.
What is their combined delay?
"""

abot(question)

'Thought: I need to find the delay for both flights FL101 and FL202, and then add them together to find the combined delay.\nAction: get_flight_delay: FL101\nPAUSE'

In [16]:
next_prompt = "Observation: {}".format(
    get_flight_delay("FL101")
)

print(next_prompt)

abot(next_prompt)

Observation: Flight FL101 was delayed by 45 minutes.


'Thought: I have the delay for flight FL101. Now, I need to find the delay for flight FL202.\nAction: get_flight_delay: FL202\nPAUSE'

In [17]:
next_prompt = "Observation: {}".format(
    get_flight_delay("FL202")
)

print(next_prompt)

abot(next_prompt)

Observation: Flight FL202 was delayed by 35 minutes.


'Thought: I have the delays for both flights. Now, I need to calculate their combined delay by adding 45 minutes and 35 minutes.\nAction: calculate: 45 + 35\nPAUSE'

In [18]:
next_prompt = "Observation: {}".format(
    calculate("45 + 35")
)

print(next_prompt)

abot(next_prompt)

Observation: 80


'Answer: The combined delay for flights FL101 and FL202 is 80 minutes.'

In [19]:
"""
This demonstrates the ReAct loop manually.

But obviously, we don't want to manually execute every action.

So now we automate it.

"""

"\nThis demonstrates the ReAct loop manually.\n\nBut obviously, we don't want to manually execute every action.\n\nSo now we automate it.\n\n"

Action parser

In [20]:
action_re = re.compile(
    r'^Action: (\w+): (.*)$'
)

This recognizes:

Action: get_flight_delay: FL101

Automated ReAct loop (This is the main block)

In [21]:
def query(question, max_turns=6):
    i = 0
    bot = Agent(prompt)
    next_prompt = question

    while i < max_turns:
        i += 1

        result = bot(next_prompt)

        print(result)

        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]

        if actions:
            action, action_input = actions[0].groups()

            if action not in known_actions:
                raise Exception(
                    "Unknown action: {}: {}".format(
                        action,
                        action_input
                    )
                )

            print(
                " -- running {} {}".format(
                    action,
                    action_input
                )
            )

            observation = known_actions[action](action_input)

            print("Observation:", observation)

            next_prompt = "Observation: {}".format(observation)

        else:
            return

Simple test

In [22]:
question = """
What is the delay of flight FL101?
"""

query(question)

Thought: I need to find the delay of flight FL101.
Action: get_flight_delay: FL101
PAUSE
 -- running get_flight_delay FL101
Observation: Flight FL101 was delayed by 45 minutes.
Answer: Flight FL101 was delayed by 45 minutes.


The main demonstration

In [23]:
question = """
Flight FL101 and Flight FL202 were both delayed.

Find their combined delay in minutes.
Then calculate what percentage of their combined flight distance
is covered by FL101.

Use the available tools to retrieve the required information
and perform the calculations.
"""

query(question)

Thought: I need to find the delay for both flights FL101 and FL202. Then, I will calculate their combined delay. After that, I will find the flight distances for both flights to calculate the percentage of the combined distance covered by FL101.

Action: get_flight_delay: FL101
PAUSE
 -- running get_flight_delay FL101
Observation: Flight FL101 was delayed by 45 minutes.
Thought: I have the delay for FL101. Now, I need to find the delay for FL202.

Action: get_flight_delay: FL202
PAUSE
 -- running get_flight_delay FL202
Observation: Flight FL202 was delayed by 35 minutes.
Thought: I have the delays for both flights. Now, I will calculate their combined delay.

Action: calculate: 45 + 35
PAUSE
 -- running calculate 45 + 35
Observation: 80
Thought: The combined delay for both flights is 80 minutes. Next, I need to find the flight distances for FL101 and FL202 to calculate what percentage of their combined flight distance is covered by FL101.

Action: get_flight_distance: FL101
PAUSE
 -- r

A second aviation example

In [24]:
question = """
Compare FL101 and FL303.

Find the difference between their delays
and the difference between their flight distances.
"""

query(question)

Thought: I need to find the delays for both FL101 and FL303 to determine the difference in their delays. Then, I need to find the flight distances for both flights to determine the difference in their distances. I'll start by finding the delay for FL101.

Action: get_flight_delay: FL101
PAUSE
 -- running get_flight_delay FL101
Observation: Flight FL101 was delayed by 45 minutes.
Thought: Now that I have the delay for FL101, I need to find the delay for FL303.

Action: get_flight_delay: FL303
PAUSE
 -- running get_flight_delay FL303
Observation: Flight FL303 was delayed by 20 minutes.
Thought: I have the delays for both flights. FL101 was delayed by 45 minutes and FL303 by 20 minutes. I will calculate the difference in their delays. After that, I will find the flight distances for both flights, starting with FL101.

Action: calculate: 45 - 20
PAUSE
 -- running calculate 45 - 20
Observation: 25
Thought: The difference in delays between FL101 and FL303 is 25 minutes. Now, I need to find t

This is a nice demonstration of the fact that the LLM determines the sequence of tool calls dynamically. I hope you learnt something new today. Keep Going! Follow my content for more useful stuff like this.

~Vishnu